# Reflection

Use a critic or reviewer loop to improve an initial answer or artifact.


## Core ideas

- Producer-critic separation: one component creates, another evaluates.
- Revision loops: critiques should point to actionable changes rather than vague preferences.
- Rubrics: reflection improves when the critic uses explicit criteria.
- Stopping conditions: use max iterations, quality thresholds, or diminishing-return checks.
- Risk: self-critique can reinforce model blind spots unless paired with tests or external evidence.


## Common failure modes



- Infinite or expensive revision loops.
- Critiques that are not tied to a rubric.
- Assuming self-critique catches factual errors without evidence.


## Framework implementation

Model reflection becomes useful when the evaluator returns typed feedback and the LangGraph loop has an explicit stopping condition.


In [ ]:
# Import the dependencies used by this example.
import os
from typing import Literal, TypedDict
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph

# Define the data shape and small operations before running them.
class Review(BaseModel):
    verdict: Literal["pass", "revise"]
    feedback: str = Field(description="Specific changes required, or why the draft passes")

class State(TypedDict):
    topic: str
    draft: str
    feedback: str
    verdict: str
    attempts: int

if not os.getenv("OPENAI_API_KEY"):
    print("Skipped: set OPENAI_API_KEY to run the evaluator-optimizer loop.")
else:
    # Configure the framework object; this line prepares it but may not execute it yet.
    model = init_chat_model("openai:gpt-5.6-sol", temperature=0)
    reviewer = model.with_structured_output(Review)

    def write(state: State):
        prompt = f"Write a concise explanation of {state['topic']}."
        if state.get("feedback"):
            prompt += f" Address this review: {state['feedback']}"
        # Execute the configured model or workflow with the input below.
        return {"draft": model.invoke(prompt).content, "attempts": state.get("attempts", 0) + 1}

    def review(state: State):
        result = reviewer.invoke(
            f"Check this draft for a definition, example, tradeoff, and verification step:\n{state['draft']}"
        )
        return {"verdict": result.verdict, "feedback": result.feedback}

    def continue_or_stop(state: State) -> Literal["write", "__end__"]:
        return "write" if state["verdict"] == "revise" and state["attempts"] < 3 else END

    builder = StateGraph(State)
    builder.add_node("write", write)
    builder.add_node("review", review)
    builder.add_edge(START, "write")
    builder.add_edge("write", "review")
    builder.add_conditional_edges("review", continue_or_stop)
    graph = builder.compile()
    result = graph.invoke({"topic": "reflection agents", "feedback": "", "attempts": 0})
    print(result["draft"])


## Offline mechanics

This version runs without credentials and exposes the control flow directly.


In [ ]:
# Define the data shape and small operations before running them.
def produce(topic):
    return f"{topic}: define the pattern and give an example."

def critique(answer):
    issues = []
    if "tradeoff" not in answer:
        issues.append("Add a tradeoff.")
    if "test" not in answer:
        issues.append("Add a verification step.")
    return issues

def revise(answer, issues):
    additions = " ".join(issues)
    return f"{answer} Revision notes: {additions}"

draft = produce("Reflection pattern")
final = revise(draft, critique(draft))
final
